# 04 Multi-Dataset Gene Alignment

This notebook aligns `GSE40732`, `GSE40888`, and `GSE64913` at the gene-symbol level. It loads local processed CSV files, creates asthma/control labels, maps probe-level expression rows to gene symbols, collapses duplicate symbols by averaging, and saves a merged common-gene matrix. It does not train models and does not apply ComBat.

> **Note:** `GSE40732` uses `data/processed/gse40732_features_with_symbols.csv`, which is created by `src/map_gse40732_refseq_to_symbols.R` and contains a `GeneSymbol` column.

## 1. Setup

Import packages, define paths, and list the datasets to align.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"
results_dir = project_root / "results"

results_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    "GSE40732": {
        "expression": processed_dir / "gse40732_expression.csv",
        "metadata": processed_dir / "gse40732_metadata.csv",
        "features": processed_dir / "gse40732_features_with_symbols.csv",
    },
    "GSE40888": {
        "expression": processed_dir / "gse40888_expression.csv",
        "metadata": processed_dir / "gse40888_metadata.csv",
        "features": processed_dir / "gse40888_features.csv",
    },
    "GSE64913": {
        "expression": processed_dir / "gse64913_expression.csv",
        "metadata": processed_dir / "gse64913_metadata.csv",
        "features": processed_dir / "gse64913_features.csv",
    },
}

merged_expression_path = processed_dir / "merged_expression_common_genes.csv"
merged_labels_path = processed_dir / "merged_labels.csv"
merged_batch_labels_path = processed_dir / "merged_batch_labels.csv"
summary_path = results_dir / "common_genes_summary.csv"

## 2. Helper Functions

These helpers standardize matrix orientation, infer labels from dataset-specific metadata, extract gene symbols from heterogeneous feature tables, and collapse duplicate symbols.

In [ ]:
def load_dataset(dataset_id, paths):
    """Load expression, metadata, and feature annotation CSVs for one dataset."""
    for kind, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing {kind} file for {dataset_id}: {path}")

    expression = pd.read_csv(paths["expression"], index_col=0)
    metadata = pd.read_csv(paths["metadata"], index_col=0)
    features = pd.read_csv(paths["features"], index_col=0, low_memory=False)

    expression = expression.apply(pd.to_numeric, errors="coerce")
    expression.index = expression.index.astype(str)
    expression.columns = expression.columns.astype(str)
    metadata.index = metadata.index.astype(str)
    features.index = features.index.astype(str)

    return expression, metadata, features


def ensure_features_by_samples(expression, metadata, dataset_id):
    """Return expression with probes/features as rows and samples as columns."""
    sample_ids = metadata.index.astype(str)
    row_matches = expression.index.astype(str).isin(sample_ids).sum()
    column_matches = expression.columns.astype(str).isin(sample_ids).sum()

    if row_matches > column_matches:
        print(f"{dataset_id}: expression appeared to be samples x features; transposing to features x samples.")
        return expression.T.copy()

    print(f"{dataset_id}: expression is treated as features x samples.")
    return expression.copy()


def metadata_text(metadata, columns=None):
    """Combine selected metadata columns into one lowercase text series."""
    if columns is None:
        columns = metadata.columns
    available = [col for col in columns if col in metadata.columns]
    if not available:
        return pd.Series("", index=metadata.index)
    return metadata[available].fillna("").astype(str).agg(" | ".join, axis=1).str.lower()


def create_labels(dataset_id, metadata):
    """Create binary asthma/control labels using dataset-specific rules."""
    if dataset_id == "GSE40732":
        cols = ["characteristics_ch1", "characteristics_ch1.1", "title", "asthma:ch1"]
        text = metadata_text(metadata, cols)
        labels = pd.Series(np.nan, index=metadata.index, dtype="object")
        labels[text.str.contains("asthma: true|_asthmatic| asthmatic|subtype: reversible|subtype: lung_function", regex=True)] = "Asthma"
        labels[text.str.contains("asthma: false|non-asthma|non asthma|control|healthy|subtype: non", regex=True)] = "Control"
    elif dataset_id == "GSE40888":
        cols = ["characteristics_ch1.3", "group:ch1", "title", "source_name_ch1"]
        text = metadata_text(metadata, cols)
        labels = pd.Series(np.nan, index=metadata.index, dtype="object")
        labels[text.str.contains("healthy control|control", regex=True)] = "Control"
        labels[text.str.contains("allergic asthmatic|non-allergic asthmatic|asthmatic|asthma", regex=True)] = "Asthma"
    elif dataset_id == "GSE64913":
        cols = ["characteristics_ch1.1", "diagnosis:ch1", "title", "source_name_ch1"]
        text = metadata_text(metadata, cols)
        labels = pd.Series(np.nan, index=metadata.index, dtype="object")
        labels[text.str.contains("healthy volunteer|healthy control|diagnosis: healthy|healthy", regex=True)] = "Control"
        labels[text.str.contains("severe asthma|severe asthmatic|diagnosis: asthma|asthma", regex=True)] = "Asthma"
    else:
        raise ValueError(f"No label logic defined for {dataset_id}")

    if labels.isna().any():
        print(f"{dataset_id}: label extraction failed for {labels.isna().sum()} samples. Metadata examples:")
        example_cols = [col for col in metadata.columns if "characteristics" in col.lower()]
        example_cols = ["title", "source_name_ch1"] + example_cols
        example_cols = [col for col in example_cols if col in metadata.columns]
        display(metadata.loc[labels.isna(), example_cols].head(15))
        raise ValueError(f"Could not create labels for all {dataset_id} samples.")

    return labels.rename("label")


def is_valid_gene_symbol(value):
    """Heuristic filter for plausible human gene symbols."""
    if pd.isna(value):
        return False
    value = str(value).strip()
    if not value or value in {"---", "--", "na", "nan", "null", "control", "pos_control", "neg_control"}:
        return False
    if re.fullmatch(r"\d+", value):
        return False
    return bool(re.search(r"[A-Za-z]", value))


def first_valid_symbol(raw_value, source_column=None):
    """Extract the first valid symbol from common multi-symbol separators."""
    if pd.isna(raw_value):
        return np.nan

    text = str(raw_value).strip()
    if not text:
        return np.nan

    # GSE40888 gene_assignment records use: accession // symbol // description ...
    if source_column == "gene_assignment" and "//" in text:
        for record in re.split(r"\s*///\s*", text):
            parts = [part.strip() for part in re.split(r"\s*//\s*", record)]
            if len(parts) >= 2 and is_valid_gene_symbol(parts[1]):
                return parts[1]

    # Other GEO platform fields often use records separated by /// and symbols separated by //.
    candidates = []
    for record in re.split(r"\s*///\s*", text):
        candidates.extend(re.split(r"\s*//\s*|\s*;\s*|\s*,\s*", record))

    for candidate in candidates:
        candidate = candidate.strip()
        if is_valid_gene_symbol(candidate):
            return candidate
    return np.nan


def extract_gene_symbols(features, dataset_id):
    """Extract one gene symbol per feature row from heterogeneous annotation columns."""
    candidate_columns = [
        "Gene Symbol",
        "gene_assignment",
        "Symbol",
        "Gene.symbol",
        "gene_symbol",
        "SYMBOL",
        "GeneSymbol",
    ]
    available = [col for col in candidate_columns if col in features.columns]

    if not available:
        print(f"{dataset_id}: no recognized gene-symbol column found.")
        print("Feature annotation columns:")
        for col in features.columns:
            print(f"- {col}")
        print("Feature annotation examples:")
        display(features.head(10))
        raise ValueError(
            f"Gene symbol extraction failed for {dataset_id}. "
            "Add a dataset-specific mapping column or preprocessing step before alignment."
        )

    best_symbols = None
    best_column = None
    best_count = -1

    for col in available:
        symbols = features[col].apply(lambda value: first_valid_symbol(value, source_column=col))
        valid_count = symbols.notna().sum()
        if valid_count > best_count:
            best_symbols = symbols
            best_column = col
            best_count = valid_count

    if best_symbols is None or best_count == 0:
        print(f"{dataset_id}: recognized columns were present but no valid symbols were extracted.")
        print("Feature annotation columns:")
        print(list(features.columns))
        display(features[available].head(10))
        raise ValueError(f"Gene symbol extraction failed for {dataset_id}.")

    print(f"{dataset_id}: using `{best_column}` for gene symbols; extracted {best_count:,} valid symbols.")
    return best_symbols.rename("gene_symbol")


def collapse_to_gene_matrix(expression_features_by_samples, features, gene_symbols, dataset_id):
    """Map probe-level rows to gene symbols and average duplicate gene symbols."""
    feature_ids = expression_features_by_samples.index.astype(str)
    feature_symbol_map = gene_symbols.copy()
    feature_symbol_map.index = features.index.astype(str)

    common_features = feature_ids.intersection(feature_symbol_map.index)
    if common_features.empty:
        raise ValueError(f"{dataset_id}: no overlap between expression row IDs and feature annotation IDs.")

    expr = expression_features_by_samples.loc[common_features].copy()
    symbols = feature_symbol_map.loc[common_features]

    valid = symbols.notna() & symbols.astype(str).str.strip().ne("")
    expr = expr.loc[valid]
    symbols = symbols.loc[valid]

    if expr.empty:
        raise ValueError(f"{dataset_id}: no expression rows remained after gene-symbol filtering.")

    expr_with_symbols = expr.copy()
    expr_with_symbols.insert(0, "gene_symbol", symbols.values)
    gene_matrix = expr_with_symbols.groupby("gene_symbol", sort=True).mean(numeric_only=True)

    print(f"{dataset_id}: collapsed {expr.shape[0]:,} mapped features to {gene_matrix.shape[0]:,} genes.")
    return gene_matrix


## 3. Load and Inspect Inputs

Load expression, metadata, and feature annotation files for each dataset. Print metadata and feature columns so dataset-specific assumptions are visible.

In [ ]:
loaded = {}

for dataset_id, paths in datasets.items():
    expression, metadata, features = load_dataset(dataset_id, paths)
    expression = ensure_features_by_samples(expression, metadata, dataset_id)

    loaded[dataset_id] = {
        "expression": expression,
        "metadata": metadata,
        "features": features,
    }

    print(f"\n{dataset_id}")
    print(f"Expression shape: {expression.shape}")
    print(f"Metadata shape: {metadata.shape}")
    print(f"Features shape: {features.shape}")
    print("Metadata columns:")
    print(list(metadata.columns))
    print("Feature annotation columns:")
    print(list(features.columns))


## 4. Create Dataset-Specific Labels

Create binary labels for each dataset. If the rules do not label all samples, print useful metadata examples and stop for review.

In [ ]:
for dataset_id, data in loaded.items():
    labels = create_labels(dataset_id, data["metadata"])
    data["labels"] = labels

    print(f"\n{dataset_id} label counts:")
    print(labels.value_counts())


## 5. Extract Gene Symbols and Collapse Probes

Extract gene symbols from feature annotation, remove missing symbols, map expression rows to symbols, and average duplicate symbols.

In [ ]:
summary_records = []

for dataset_id, data in loaded.items():
    expression = data["expression"]
    metadata = data["metadata"]
    features = data["features"]
    labels = data["labels"]

    gene_symbols = extract_gene_symbols(features, dataset_id)
    mapped_gene_symbol_count = int(gene_symbols.notna().sum())
    gene_matrix = collapse_to_gene_matrix(expression, features, gene_symbols, dataset_id)
    data["gene_matrix"] = gene_matrix

    summary_records.append(
        {
            "dataset": dataset_id,
            "samples": len(labels),
            "asthma_samples": int((labels == "Asthma").sum()),
            "control_samples": int((labels == "Control").sum()),
            "features_before_gene_mapping": expression.shape[0],
            "gene_symbols_mapped": mapped_gene_symbol_count,
            "genes_after_collapsing": gene_matrix.shape[0],
        }
    )


## 6. Align Common Genes Across Datasets

Find the intersection of gene symbols shared by all three datasets and merge the aligned matrices into one sample-by-gene table.

In [ ]:
common_genes = sorted(
    set.intersection(*(set(data["gene_matrix"].index) for data in loaded.values()))
)

if not common_genes:
    raise ValueError("No common gene symbols were found across all datasets.")

merged_expression_parts = []
merged_label_parts = []
merged_batch_parts = []

for dataset_id, data in loaded.items():
    gene_matrix = data["gene_matrix"].loc[common_genes]
    sample_matrix = gene_matrix.T.copy()
    sample_matrix.index = sample_matrix.index.astype(str)

    labels = data["labels"].copy()
    labels.index = labels.index.astype(str)
    common_samples = sample_matrix.index.intersection(labels.index)

    sample_matrix = sample_matrix.loc[common_samples]
    labels = labels.loc[common_samples]

    # Prefix sample IDs to keep them unique after merging datasets.
    sample_matrix.index = [f"{dataset_id}_{sample_id}" for sample_id in sample_matrix.index]
    labels.index = sample_matrix.index

    batch_labels = pd.Series(dataset_id, index=sample_matrix.index, name="dataset")

    merged_expression_parts.append(sample_matrix)
    merged_label_parts.append(labels.rename("label"))
    merged_batch_parts.append(batch_labels)

merged_expression = pd.concat(merged_expression_parts, axis=0)
merged_labels = pd.concat(merged_label_parts, axis=0).to_frame()
merged_batch_labels = pd.concat(merged_batch_parts, axis=0).to_frame()

merged_expression.index.name = "sample_id"
merged_labels.index.name = "sample_id"
merged_batch_labels.index.name = "sample_id"

print(f"Common genes across all datasets: {len(common_genes):,}")
print(f"Final merged expression matrix shape: {merged_expression.shape}")


## 7. Save Aligned Outputs

Save the merged expression matrix, sample labels, batch labels, and common-gene summary for future preprocessing and batch-correction work.

In [ ]:
summary_df = pd.DataFrame(summary_records)
summary_df["common_genes_across_all_datasets"] = len(common_genes)
summary_df["final_merged_samples"] = merged_expression.shape[0]
summary_df["final_merged_genes"] = merged_expression.shape[1]

merged_expression.to_csv(merged_expression_path)
merged_labels.to_csv(merged_labels_path)
merged_batch_labels.to_csv(merged_batch_labels_path)
summary_df.to_csv(summary_path, index=False)

print(f"Saved merged expression matrix to {merged_expression_path}")
print(f"Saved merged labels to {merged_labels_path}")
print(f"Saved merged batch labels to {merged_batch_labels_path}")
print(f"Saved common genes summary to {summary_path}")


## 8. Alignment Summary

Print the requested summary for dataset sizes, class balance, gene counts, common genes, and final merged matrix shape.

In [ ]:
print("Samples and class counts per dataset:")
display(summary_df[["dataset", "samples", "asthma_samples", "control_samples"]])

print("\nGene symbols mapped and genes before/after alignment per dataset:")
display(summary_df[["dataset", "features_before_gene_mapping", "gene_symbols_mapped", "genes_after_collapsing", "common_genes_across_all_datasets"]])

print(f"\nNumber of common genes across all datasets: {len(common_genes):,}")
print(f"Final merged matrix shape: {merged_expression.shape}")

print("\nMerged label counts:")
print(merged_labels["label"].value_counts())

print("\nMerged batch counts:")
print(merged_batch_labels["dataset"].value_counts())
